# 02a — Build & Validate the Agent

Imports the shared `agent_lib.py` module, unit-tests each of the three tools directly (without an LLM), then assembles the ReAct agent and executes a smoke-test invocation.

### Validation Steps

- Import the shared `agent_lib.py` library
- Unit-test semantic retrieval against the Vector Search index
- Unit-test conflict checking against `lexpath_conflicts`
- Unit-test case routing through `lexpath_routing_schema`
- Assemble the full ReAct agent
- Execute a single end-to-end smoke test to verify tool orchestration

### Notes

Databricks notebooks do not share Python runtime state. As a result, this notebook focuses solely on build validation. The downstream notebooks (`02b_run_agent` and `03_evaluation`) independently reconstruct the agent from the same shared library to ensure reproducibility and avoid cross-notebook dependency issues.

In [0]:
# Configure Widgets
dbutils.widgets.text("catalog", "workspace", "Unity Catalog name")
dbutils.widgets.text("schema", "default", "Schema")
dbutils.widgets.text("vs_endpoint", "lexpath_vs_endpoint", "Vector Search Endpoint")
dbutils.widgets.text("llm_endpoint", "anthropic-claude-sonnet-4-6", "LLM Serving Endpoint")

In [0]:
# Install LangChain/Databricks/Vector Search/MLflow Stack
%pip install --upgrade 'langchain>=1.3.0' langgraph databricks-langchain databricks-vectorsearch mlflow

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Restart Python
dbutils.library.restartPython()

In [0]:
# Import libraries and configure the agent
import sys, os, json
import importlib
sys.path.append(os.getcwd())  # ensure the notebook's workspace folder is importable
 
import mlflow
import agent_lib
importlib.reload(agent_lib)  # Reload to pick up any file changes
 
agent_lib.configure(
    catalog=dbutils.widgets.get("catalog"),
    schema=dbutils.widgets.get("schema"),
    vs_endpoint=dbutils.widgets.get("vs_endpoint"),
)
mlflow.langchain.autolog()

agent_lib configured — index workspace.default.ledgar_provisions_index, 100 routing labels


####Tool 1: Semantic Retrieval

In [0]:
# Unit Test Tool 1: semantic retrieval should return TOP_K grounded provisions
out = json.loads(agent_lib.semantic_retrieval.invoke(
    {"query": "My employer is refusing to pay the severance in my contract"}))
assert len(out) > 0 and {"category", "score", "provision_excerpt"} <= set(out[0]), out
print(f"✅ semantic_retrieval: {len(out)} results, top category = {out[0]['category']}")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
✅ semantic_retrieval: 5 results, top category = Benefits


Trace(trace_id=tr-0ce6d18b6a85fa7c3893e20251d1b264)

####Tool 2: Conflict Check

In [0]:
# Unit Test Tool 2 — conflict check: known client → CONFLICT_FLAG, unknown name → CLEARED
flag = json.loads(agent_lib.conflict_check.invoke({"party_names": "Atlas Manufacturing"}))
clear = json.loads(agent_lib.conflict_check.invoke({"party_names": "Jane Doe"}))
assert flag["result"] == "CONFLICT_FLAG" and clear["result"] == "CLEARED", (flag, clear)
print(f"✅ conflict_check: flagged {len(flag['matches'])} matters; unknown name CLEARED")

✅ conflict_check: flagged 2 matters; unknown name CLEARED


[Trace(trace_id=tr-64caa22a62cf297ea7d128c3b656e60a), Trace(trace_id=tr-72db81808f9c6451ef622bb21bffa47a)]

####Tool 3: Case Routing

In [0]:
# Unit Test Tool 3 — case routing: valid label routes, junk label → Unrouted
routed = json.loads(agent_lib.case_routing.invoke({"category_label": "Governing Laws"}))
junk = json.loads(agent_lib.case_routing.invoke({"category_label": "Not A Real Label"}))
assert routed["recognized"] and not junk["recognized"], (routed, junk)
print(f"✅ case_routing: 'Governing Laws' → {routed['practice_area']}; junk → Unrouted")

✅ case_routing: 'Governing Laws' → Litigation; junk → Unrouted


[Trace(trace_id=tr-fde7c4add2201806e8984d5d5c5d0a11), Trace(trace_id=tr-20f67a15f683bab5d5db0f3e47bea529)]

In [0]:
# Assemble the Agent
LLM_ENDPOINT = dbutils.widgets.get("llm_endpoint")
executor = agent_lib.build_agent(LLM_ENDPOINT, verbose=True)
print(f"Agent assembled on {LLM_ENDPOINT} with {len(agent_lib.tools)} tools")

Agent assembled on anthropic-claude-sonnet-4-6 with 3 tools


In [0]:
# End-to-end test (run the Reason → Act → Observe loop with verbose=True)
result = executor.invoke({"input":
    "My business partner and I signed an agreement that says disputes go to arbitration, "
    "but now they filed a lawsuit in court instead. I want to enforce the arbitration clause."})
 
profile = agent_lib.extract_json(result.get("output", ""))
assert profile.get("status") == "READY_FOR_REVIEW", profile
assert profile.get("predicted_category"), profile
print(json.dumps(profile, indent=2))

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
{
  "status": "READY_FOR_REVIEW",
  "issue_summary": "The client entered into a business agreement containing an arbitration clause requiring disputes to be resolved exclusively through arbitration. Despite this clause, the client's business partner initiated litigation in court. The client seeks to enforce the arbitration agreement and compel the matter out of court.",
  "predicted_category": "Arbitration",
  "practice_area": "Litigation",
  "parties": [],
  "conflict_status": "NOT_RUN",
  "conflict_matches": [],
  "clarifying_questions": [],
  "routing_rationale": "Multiple retrieved LEDGAR provisions closely match this matter, including clauses stating disputes 'shall be settled exclusively by arbitration' and that arbitration awards may be enforced in any court with jurisdict

Trace(trace_id=tr-03c837ce6a3419b573911fb91ebe0b35)

## Build Summary

- All three tools were validated directly against their backing resources:
  - Vector Search index
  - `lexpath_conflicts`
  - `lexpath_routing_schema`
- The agent was assembled via `agent_lib.build_agent()` and successfully smoke-tested end-to-end.
- The resulting trace is available in this notebook's **MLflow Experiment → Traces** tab.
